# 02 · First pipeline: YOLO11 baseline and a valid submission

Goal: a complete, simple pipeline end to end (train → validate with the official COCO metric → submission),
so we beat the baseline early. This model is also the starting point of the **anchor-free** section.

**Run on Kaggle:** GPU T4, Internet ON. Inputs: the competition data + the output of notebook 01.

In [1]:
%%writefile vd_utils.py
"""Shared helpers for the VisDrone-VID homework: data loading, COCO evaluation, submission writing."""
import json
from pathlib import Path

import numpy as np
import pandas as pd

CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]


def find_data_root(start="/kaggle/input"):
    """Folder that contains train/, val/ and test/ (found by its train/annotations_coco.json)."""
    hits = sorted(Path(start).glob("**/train/annotations_coco.json"))
    if not hits:
        raise FileNotFoundError(f"train/annotations_coco.json not found under {start}: attach the competition data")
    return hits[0].parent.parent


def load_split(root, split):
    """COCO json of one split -> (images, annotations) DataFrames with 0-based `cls` and pixel boxes."""
    root = Path(root)
    coco = json.loads((root / split / "annotations_coco.json").read_text())
    imgs = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
    if "video" in imgs:
        imgs["clip"] = imgs["video"].astype(str)
    else:
        imgs["clip"] = imgs["file_name"].map(lambda f: Path(f).parent.name)
    imgs["path"] = [_image_path(root, split, f) for f in imgs["file_name"]]

    anns = pd.DataFrame(coco["annotations"])
    anns[["x", "y", "w", "h"]] = pd.DataFrame(anns["bbox"].tolist(), index=anns.index)
    anns["iscrowd"] = anns.get("iscrowd", 0)
    anns["cls"] = anns["category_id"] - 1
    anns = anns.merge(imgs[["image_id", "clip", "width", "height"]], on="image_id")
    return imgs, anns


def _image_path(root, split, file_name):
    for p in (root / split / file_name, root / split / "images" / file_name, root / file_name):
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"image {file_name} not found in {root / split}")


def coco_eval(gt_json, dets, img_ids=None, cat_ids=None, verbose=False):
    """COCO mAP with ignored regions (iscrowd=1) skipped and up to 500 detections per image.

    dets: list of dicts {image_id, category_id (1-based), bbox [x, y, w, h], score}.
    """
    import contextlib, io
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    quiet = contextlib.redirect_stdout(io.StringIO()) if not verbose else contextlib.nullcontext()
    with quiet:
        gt = COCO(str(gt_json))
        if not dets:
            return {k: 0.0 for k in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]}
        dt = gt.loadRes(dets)
        E = COCOeval(gt, dt, "bbox")
        E.params.maxDets = [1, 100, 500]
        if img_ids is not None:
            E.params.imgIds = list(img_ids)
        if cat_ids is not None:
            E.params.catIds = list(cat_ids)
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    out = {k: float(v) for k, v in zip(["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"], s[:6])}
    # per-class AP[.5:.95] from the same evaluation: precision has shape [IoU, recall, class, area, maxDets]
    prec = E.eval["precision"][:, :, :, 0, -1]
    for k, cat in enumerate(E.params.catIds):
        p = prec[:, :, k]
        out[f"AP_{CLASSES[cat - 1]}"] = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
    return out


def write_submission(dets, test_csv, out_path, max_dets=500):
    """dets: dict image_id (str, e.g. clip_001_0000001) -> array N x 6 [cls, score, x, y, w, h] in pixels."""
    test = pd.read_csv(test_csv)
    rows = []
    for image_id in test["image_id"]:
        d = np.asarray(dets.get(image_id, np.zeros((0, 6))), dtype=float).reshape(-1, 6)
        d = d[np.argsort(-d[:, 1])][:max_dets]
        rows.append(" ".join(f"{int(c)} {s:.4f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}" for c, s, x, y, w, h in d))
    sub = pd.DataFrame({"image_id": test["image_id"], "PredictionString": rows})
    sub.to_csv(out_path, index=False)
    return sub

Writing vd_utils.py


In [2]:
# install and imports
!pip install -q ultralytics pycocotools
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, time, json, random
from pathlib import Path
import numpy as np, pandas as pd, torch
from ultralytics import YOLO

YOLO_DIR = next(Path("/kaggle/input").glob("**/vd_yolo/data.yaml")).parent      # output of notebook 01
import vd_utils as vd
ROOT, WORK = vd.find_data_root(), Path("/kaggle/working")
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("data:", ROOT, "| yolo dataset:", YOLO_DIR, "| GPU:", torch.cuda.get_device_name(0))

# the yaml written by notebook 01 points to /kaggle/working; point it to the read-only input copy
DATA_YAML = WORK / "data.yaml"
DATA_YAML.write_text(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnames: {dict(enumerate(vd.CLASSES))}\n")
val_imgs, _ = vd.load_split(ROOT, "val")
VAL_GT = ROOT / "val" / "annotations_coco.json"

data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 | yolo dataset: /kaggle/input/notebooks/yarema1234/notebook0f34d49e70/vd_yolo | GPU: Tesla T4


## 1. Training setup (shared protocol for all models)
* train: every 4th frame of the 56 train clips (6 072 frames); validation: the 7 val clips, all frames
* input size 1280 px: test frames are mostly 1360–1920 px wide, and a quarter of the objects are under 32 px,
  so 640 px would shrink many of them to a few pixels
* pretrained COCO weights, SGD default schedule with cosine LR, default YOLO augmentation (mosaic off for the last 2 epochs)

In [3]:
CFG = dict(model="yolo11s.pt", imgsz=1280, epochs=12, batch=8, cos_lr=True, close_mosaic=2, seed=SEED,
           workers=4, device=0, project=str(WORK / "runs"), name="yolo11s_1280", exist_ok=True, plots=True)

# reuse weights from an earlier run of this notebook if its output is attached as input (saves ~1.8 GPU hours)
prev = sorted(Path("/kaggle/input").glob(f"**/{CFG['name']}/weights/best.pt"))
if prev:
    best = prev[0]; print("reusing trained weights:", best)
else:
    t0 = time.time()
    model = YOLO(CFG["model"])
    model.train(data=str(DATA_YAML), **{k: v for k, v in CFG.items() if k != "model"})
    print(f"training time: {(time.time() - t0) / 60:.0f} min")
    best = WORK / "runs" / CFG["name"] / "weights" / "best.pt"
    del model
import gc; gc.collect(); torch.cuda.empty_cache()    # free the training model before inference

reusing trained weights: /kaggle/input/notebooks/yarema1234/notebooka290378ae2/runs/yolo11s_1280/weights/best.pt


## 2. Validation with the official metric
YOLO's own validation runs on the masked val images. The numbers we report come from `vd.coco_eval` on the
**original** val frames and the untouched COCO json: ignored regions are `iscrowd=1` and are skipped by the evaluator,
up to 500 detections per image, low confidence threshold (0.001) as usual for mAP.

In [4]:
def predict(model, paths, imgsz=1280, conf=0.001, iou=0.6, bs=4):
    # returns {path: array N x 6 [cls, score, x, y, w, h]} in original pixels, and ms per image.
    # Images are passed in small explicit chunks: a long list would be processed as one huge batch (GPU OOM).
    out, t0 = {}, time.time()
    for k in range(0, len(paths), bs):
        for r in model.predict(paths[k:k + bs], imgsz=imgsz, conf=conf, iou=iou, max_det=500, verbose=False):
            b = r.boxes; xyxy = b.xyxy.cpu().numpy()
            out[r.path] = np.c_[b.cls.cpu().numpy(), b.conf.cpu().numpy(), xyxy[:, :2], xyxy[:, 2:] - xyxy[:, :2]]
    return out, (time.time() - t0) / len(paths) * 1000

def to_coco(preds, imgs):
    pid = dict(zip(imgs["path"], imgs["image_id"]))
    return [{"image_id": int(pid[p]), "category_id": int(c) + 1, "bbox": [float(x), float(y), float(w), float(h)], "score": float(s)}
            for p, d in preds.items() for c, s, x, y, w, h in d]

model = YOLO(best)
val_preds, ms = predict(model, val_imgs["path"].tolist())
res = vd.coco_eval(VAL_GT, to_coco(val_preds, val_imgs))
res["ms / image"] = ms
RESULTS = pd.DataFrame([{"model": "YOLO11s 1280 (baseline)", **res}]).set_index("model")
RESULTS.to_csv(WORK / "results_02.csv")
main = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms / image"]
display(RESULTS[main].round(4))

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,ms / image
model,,,,,,,
YOLO11s 1280 (baseline),0.1984,0.3949,0.1739,0.1077,0.1842,0.3619,74.5434


## 3. Per-class AP

In [5]:
# per-class AP[.5:.95], taken from the same COCO evaluation
display(RESULTS[[f"AP_{c}" for c in vd.CLASSES]].T.round(3))

model,YOLO11s 1280 (baseline)
AP_person,0.278
AP_bicycle,0.226
AP_car,0.424
AP_van,0.205
AP_truck,0.106
AP_tricycle,0.169
AP_bus,0.066
AP_motor,0.118


## 4. Test predictions and submission

In [6]:
test = pd.read_csv(ROOT / "test_images.csv")
test["path"] = [str(ROOT / f) for f in test["file_name"]]
test_preds, _ = predict(model, test["path"].tolist())
dets = {i: test_preds.get(p, np.zeros((0, 6))) for i, p in zip(test["image_id"], test["path"])}
sub = vd.write_submission(dets, ROOT / "test_images.csv", WORK / "submission.csv")
print(sub.head(3)); print("boxes per image:", np.mean([len(d) for d in dets.values()]).round(1))

           image_id                                   PredictionString
0  clip_001_0000001  0 0.7653 629.5 916.6 65.8 160.2 0 0.7606 720.3...
1  clip_001_0000002  0 0.7714 1657.1 793.7 68.8 152.3 0 0.7572 1713...
2  clip_001_0000003  0 0.7619 1721.5 800.4 71.8 151.3 0 0.7568 616....
boxes per image: 418.6
